In [1]:
import pandas as pd
import numpy as np

In [18]:
df = pd.read_csv("../Raw/WS_NA_SEC_DSS_csv_col.csv")
df

/var/folders/_x/74827dzs033cwdj2j4gch59r0000gn/T/ipykernel_51777/470624538.py:1: DtypeWarning: Columns (0: COMMENT_TS, 1: DATA_COMP) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../Raw/WS_NA_SEC_DSS_csv_col.csv")


,FREQ,Frequency,ADJUSTMENT,Adjustment indicator,REF_AREA,Reference area,COUNTERPART_AREA,Counterpart area,REF_SECTOR,Reporting institutional sector,...,2023-Q3,2023-Q4,2024,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3
0,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,CA,Canada,XW,World,S122,"Deposit taking corporations, except the Centra...",...,460.997967,473.965110,NaN,459.399795,469.437853,493.606670,483.597089,504.275034,507.511932,517.407872
1,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,FR,France,XW,World,S12Q,Insurance corporations and Pension Funds,...,0.000000,0.352714,NaN,1.009149,1.953463,1.956109,1.963218,1.967765,1.948949,1.954959
2,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,CA,Canada,XW,World,S122,"Deposit taking corporations, except the Centra...",...,740.629072,745.290053,NaN,711.358331,679.017183,680.443727,636.305192,614.700406,618.211747,603.846389
3,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,DK,Denmark,XW,World,S122,"Deposit taking corporations, except the Centra...",...,842.081965,840.695000,NaN,825.332278,793.771278,779.408263,790.711161,834.521539,809.572797,794.525087
4,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,GR,Greece,XW,World,S1M,Households and non profit institutions serving...,...,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70317,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,AR,Argentina,XW,World,S12P,Other financial institutions (Financial corpor...,...,84.541000,233.290000,NaN,18.061000,200.021000,395.262000,682.671000,498.411000,1237.838000,725.084000
70318,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,PL,Poland,XW,World,S12P,Other financial institutions (Financial corpor...,...,0.238537,0.033190,NaN,-0.934192,0.009454,0.009583,0.009531,0.009254,0.006390,0.009536
70319,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,BE,Belgium,XW,World,S1,Total economy,...,-0.056071,0.348886,NaN,1.033245,-0.437522,2.892457,0.751894,0.717012,0.355189,-0.180877
70320,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,EE,Estonia,XW,World,S121,Central bank,...,0.000000,0.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [19]:
df["Frequency"].value_counts()

Frequency
Quarterly    69930
Annual         392
Name: count, dtype: int64

In [20]:
cols = ['ADJUSTMENT', 'REF_SECTOR', 'COUNTERPART_AREA', 'COUNTERPART_SECTOR', 
        'CONSOLIDATION', 'ACCOUNTING_ENTRY', 'STO', 'INSTR_ASSET', 
        'MATURITY', 'UNIT_MEASURE', 'VALUATION']

for col in cols:
    print(f"\n{col}: (10 first values)")
    print(df[col].unique().tolist()[:10])


ADJUSTMENT: (10 first values)
['N']

REF_SECTOR: (10 first values)
['S122', 'S12Q', 'S1M', 'S121', 'S12', 'S1', 'S1311', 'S13', 'S12P', 'S125A']

COUNTERPART_AREA: (10 first values)
['XW', '1E', '5Z']

COUNTERPART_SECTOR: (10 first values)
['S1', 'S13', 'S11', 'S1M', 'S12']

CONSOLIDATION: (10 first values)
['N']

ACCOUNTING_ENTRY: (10 first values)
['L', 'A']

STO: (10 first values)
['LE', 'F']

INSTR_ASSET: (10 first values)
['F3', 'F3VRC', 'F3VRB', 'F3FR', 'F3VR', 'F3VRA', 'F3B']

MATURITY: (10 first values)
['T', 'Y25', 'Y5A', 'L', 'Y12', 'LS', 'S', 'YA_', 'TT']

UNIT_MEASURE: (10 first values)
['USD', 'EUR', 'DKK', 'THB', 'TRY', 'ZAR', 'HUF', 'ILS', 'SEK', 'CAD']

VALUATION: (10 first values)
['N', 'F', 'M']


In [22]:
mask = (
    (df['ACCOUNTING_ENTRY'] == 'L')   # liabilities = securities issued (not assets)
    & (df['STO'] == 'LE')             # stocks (amounts outstanding), not flows (F)
    & (df['REF_SECTOR'] == 'S1')      # all sectors combined (total economy)
    & (df['COUNTERPART_AREA'] == 'XW') # all counterparts (world) = domestic + international
    & (df['INSTR_ASSET'] == 'F3')     # total debt securities (not subcategories)
    & (df['MATURITY'] == 'T')         # total maturity (short + long combined)
    & (df['VALUATION'] == 'M')        # market value (use 'N' for nominal if you prefer)
    & (df['UNIT_MEASURE'] == 'USD')   # standardise to USD
)

df_debt = df[mask]
df_debt

,FREQ,Frequency,ADJUSTMENT,Adjustment indicator,REF_AREA,Reference area,COUNTERPART_AREA,Counterpart area,REF_SECTOR,Reporting institutional sector,...,2023-Q3,2023-Q4,2024,2024-Q1,2024-Q2,2024-Q3,2024-Q4,2025-Q1,2025-Q2,2025-Q3
13253,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,FR,France,XW,World,S1,Total economy,...,626.584707,682.064539,NaN,682.947267,700.683939,739.375718,704.423396,724.050324,757.819664,763.860631
13265,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,U2,Euro area (Member States and Institutions of t...,XW,World,S1,Total economy,...,18903.705356,20638.006701,NaN,20535.086239,20399.934721,22047.432320,20420.771934,21431.017399,23834.464466,24032.294012
13373,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,GR,Greece,XW,World,S1,Total economy,...,136.074838,151.739648,NaN,151.823845,151.084575,165.056207,156.651592,162.720807,180.214067,181.007421
13395,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,GB,United Kingdom,XW,World,S1,Total economy,...,671.367462,720.776736,NaN,717.536039,730.508196,733.116140,730.785471,740.651650,796.006296,816.815412
13845,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,SE,Sweden,XW,World,S1,Total economy,...,315.714727,337.179953,NaN,337.291053,329.202880,363.043626,338.990340,361.969686,397.884653,392.355920
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58632,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,AT,Austria,XW,World,S1,Total economy,...,590.426209,638.688619,NaN,646.329567,640.764362,692.580379,638.666417,681.496544,746.552841,759.611706
58773,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,BR,Brazil,XW,World,S1,Total economy,...,2841.984192,3112.043786,NaN,3069.143498,2889.471105,2975.901759,2671.559815,2980.413261,3285.976797,3495.301643
58863,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,BG,Bulgaria,XW,World,S1,Total economy,...,18.435437,22.890214,NaN,22.525766,22.993695,27.812526,26.408053,27.647186,34.785478,39.334596
58878,Q,Quarterly,N,Neither seasonally adjusted nor calendar adjus...,PL,Poland,XW,World,S1,Total economy,...,331.594471,389.048733,NaN,406.319503,419.978682,463.470300,446.247129,509.988026,572.559124,583.194360


In [37]:
# Melt the dataframe
qtr_columns = [col for col in df_debt.columns if (col.startswith("19") or col.startswith("20")) and (str(col) + "00000")[5] == "Q"]

df_debt_melted = df_debt.melt(id_vars=['REF_AREA', 'Reference area', 'COUNTERPART_AREA', 'Counterpart area'], value_vars=qtr_columns, var_name='Date', value_name='Value')
df_debt_melted

,REF_AREA,Reference area,COUNTERPART_AREA,Counterpart area,Date,Value
0,FR,France,XW,World,1946-Q1,NaN
1,U2,Euro area (Member States and Institutions of t...,XW,World,1946-Q1,NaN
2,GR,Greece,XW,World,1946-Q1,NaN
3,GB,United Kingdom,XW,World,1946-Q1,NaN
4,SE,Sweden,XW,World,1946-Q1,NaN
...,...,...,...,...,...,...
92824,AT,Austria,XW,World,2025-Q3,759.611706
92825,BR,Brazil,XW,World,2025-Q3,3495.301643
92826,BG,Bulgaria,XW,World,2025-Q3,39.334596
92827,PL,Poland,XW,World,2025-Q3,583.194360


In [38]:
# Now get the iso3 from pycountry
# map = iso3: country name
import pycountry

def get_iso3(country_name):
    try:
        return pycountry.countries.lookup(country_name).alpha_3
    except LookupError:
        return None

df_debt_melted["iso3"] = df_debt_melted["Reference area"].apply(get_iso3)
df_debt_melted

,REF_AREA,Reference area,COUNTERPART_AREA,Counterpart area,Date,Value,iso3
0,FR,France,XW,World,1946-Q1,NaN,FRA
1,U2,Euro area (Member States and Institutions of t...,XW,World,1946-Q1,NaN,NaN
2,GR,Greece,XW,World,1946-Q1,NaN,GRC
3,GB,United Kingdom,XW,World,1946-Q1,NaN,GBR
4,SE,Sweden,XW,World,1946-Q1,NaN,SWE
...,...,...,...,...,...,...,...
92824,AT,Austria,XW,World,2025-Q3,759.611706,AUT
92825,BR,Brazil,XW,World,2025-Q3,3495.301643,BRA
92826,BG,Bulgaria,XW,World,2025-Q3,39.334596,BGR
92827,PL,Poland,XW,World,2025-Q3,583.194360,POL


In [39]:
# Remove nan iso3
df_debt_melted = df_debt_melted.dropna(subset=["iso3"])

In [41]:
# Get last quarter as last measurement for that year
df_debt_melted["year"] = df_debt_melted["Date"].str[:4]
df_debt_melted["quarter"] = df_debt_melted["Date"].str[5:]
df_debt_melted = df_debt_melted[df_debt_melted["quarter"] == "Q4"]
df_debt_melted

,REF_AREA,Reference area,COUNTERPART_AREA,Counterpart area,Date,Value,iso3,Year,Quarter,year,quarter
873,FR,France,XW,World,1946-Q4,NaN,FRA,1946,Q4,1946,Q4
875,GR,Greece,XW,World,1946-Q4,NaN,GRC,1946,Q4,1946,Q4
876,GB,United Kingdom,XW,World,1946-Q4,NaN,GBR,1946,Q4,1946,Q4
877,SE,Sweden,XW,World,1946-Q4,NaN,SWE,1946,Q4,1946,Q4
878,ES,Spain,XW,World,1946-Q4,NaN,ESP,1946,Q4,1946,Q4
...,...,...,...,...,...,...,...,...,...,...,...
91951,AT,Austria,XW,World,2024-Q4,638.666417,AUT,2024,Q4,2024,Q4
91952,BR,Brazil,XW,World,2024-Q4,2671.559815,BRA,2024,Q4,2024,Q4
91953,BG,Bulgaria,XW,World,2024-Q4,26.408053,BGR,2024,Q4,2024,Q4
91954,PL,Poland,XW,World,2024-Q4,446.247129,POL,2024,Q4,2024,Q4


In [42]:
# rename Value to value
df_debt_melted = df_debt_melted.rename(columns={"Value": "value"})

In [43]:
# Restict to columns required
df_debt_melted = df_debt_melted[["year", "iso3", "value"]]
df_debt_melted

,year,iso3,value
873,1946,FRA,NaN
875,1946,GRC,NaN
876,1946,GBR,NaN
877,1946,SWE,NaN
878,1946,ESP,NaN
...,...,...,...
91951,2024,AUT,638.666417
91952,2024,BRA,2671.559815
91953,2024,BGR,26.408053
91954,2024,POL,446.247129


In [44]:
df_debt_melted.to_csv("../Clean/BIS_Debt.csv", index=False)